In [ ]:
# 0) Colab: mount Drive (train-data + runs) + clone code từ GitHub
# Chạy được bằng BẤT KỲ account nào (chính HOẶC phụ) — dùng CHUNG 1 path, không sửa code khi đổi account.
#
# YÊU CẦU 1 LẦN cho account phụ (account chính bỏ qua):
#   Mở link share (quyền Editor) -> chuột phải folder "recommender_train_colab"
#   -> Organize/Sắp xếp -> Add shortcut to Drive (Thêm lối tắt) -> My Drive.
#   Shortcut giữ nguyên tên => path /content/drive/MyDrive/recommender_train_colab
#   giống hệt account chính. Editor => ghi checkpoint/runs.csv thẳng vào folder gốc.
#
# ⚠ TRAIN-DATA V2: Drive phải chứa train-data ĐÃ REBUILD (cold_items + eval_seen +
#   examples {val,test}_cold + history full). Bản v1 KHÔNG chạy được với code v2.
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

DRIVE_DIR = '/content/drive/MyDrive/recommender_train_colab'
assert os.path.isdir(DRIVE_DIR), (
    f"Khong thay {DRIVE_DIR}.\n"
    "Account phu: mo link share (Editor) trong Drive roi 'Add shortcut to My Drive' "
    "cho folder recommender_train_colab. Account chinh: kiem tra folder con o My Drive."
)

REPO = 'https://github.com/CrocodixD/anime-recommender.git'
CODE = '/content/recommender'

if os.path.exists(CODE):
    !cd {CODE} && git pull
else:
    !git clone {REPO} {CODE}

# train-data nằm ở retriever/train-data (ROOT của src/config.py = retriever/)
if not os.path.exists(f'{CODE}/retriever/train-data'):
    !cp -r "{DRIVE_DIR}/train-data" "{CODE}/retriever/train-data"

%cd {CODE}

In [ ]:
# 1) Imports + path (code train ở retriever/src — import flat)
# imp shim: vài lib trên Colab (py3.12) còn import 'imp' đã bị gỡ khỏi stdlib
import sys, importlib
sys.modules['imp'] = importlib

import pathlib
HERE = pathlib.Path.cwd()
SRC_DIR = HERE / 'retriever' / 'src' if (HERE / 'retriever' / 'src').exists() else HERE
sys.path.insert(0, str(SRC_DIR))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import config, data, model as M, loss, metrics, train
print('torch', torch.__version__)

In [ ]:
# 2) Đường dẫn persistence trên Drive (checkpoint + history + leaderboard)
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/recommender_train_colab')
VERSION = 'v5'                                # v5 = data v2 + protocol v2 (KHÔNG so số với v1-v4)
RUNS_DIR = DRIVE / 'runs' / VERSION            # mỗi run: runs/<VERSION>/<run_name>/best.pt + history.json
RESULTS_CSV = DRIVE / 'runs.csv'               # leaderboard CHUNG mọi version (có cột 'version')
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('version', VERSION, '· runs ->', RUNS_DIR)

In [ ]:
# 3) Base config T4 (Colab free: 2 core / 13GB RAM / 15GB VRAM). ckpt_dir set per-run ở helper (cell 5).
# PROTOCOL V2: headline = warm recall@200 (val); eval_ks tới 500; mask seen−query.
# Mọi knob mới có flag bật/tắt (search lật được) — xem retriever/CLAUDE.md §Config surface.
base_cfg = config.TwoTowerConfig(
    # model
    d=128,
    history_source='cache',      # 'embed' = bảng history trainable (đòn bẩy chính cần thử)

    # synopsis (content text-emb item-side; CẦN synopsis_emb.npy + synopsis_low_info.npy trong train-data)
    use_synopsis=False,          # True = bật nhánh synopsis (sinh local bằng 07 rồi đẩy 2 .npy lên Drive train-data/)
    synopsis_dim=48,             # chiều sau chiếu; synopsis_proj_hidden=[] (Linear thuần) | [128] nếu underfit
    synopsis_normalize='none',   # artifact đã L2 sẵn ('l2'/'standardize' nếu đổi cách chuẩn hoá)

    # loss
    tau=0.07,
    beta=1.0,
    logq_alpha=1.0,              # hệ số logQ (thử 0.5/0.75 — metric thưởng popularity)

    # train
    lr=1e-3,
    optimizer='adam',           # 'adamw' = decouple weight_decay (đòn bẩy regularization khi search)
    weight_decay=0.0,
    batch_size=8192,
    epochs=1,
    hist_dropout=0.12,
    m_hardneg=5,
    train_hist_len=32,           # sample history/anchor từ FULL list (augmentation)
    max_examples_per_user=None,  # vd 64: epoch ngắn ~4x + cân trọng số per-user
    # train_user_frac=0.15,      # HP-search COARSE: chỉ giữ % user (full catalog+logQ+eval không đổi); None = full

    # eval (protocol v2)
    eval_every_steps=500,

    # infra
    num_workers=2,
    # subset=200_000,            # bỏ comment để chạy thử nhanh; None = full
)
print('device', base_cfg.device)   # 'cuda' trên T4
base_cfg

In [ ]:
# 4) Build module (sanity-check shape / param count)
spec, logq, item_table, users, mdl = train.build(base_cfg)
n_params = sum(p.numel() for p in mdl.parameters() if p.requires_grad)
print(f'num_items={item_table.num_items}  num_users={users.num_users}  params={n_params:,}')
print('logq', logq.shape, 'finite', int(torch.isfinite(logq).sum()))

In [ ]:
# 5) Helpers thử nghiệm: mỗi run có tên riêng -> best.pt/history.json/row.json/config.json.
# GHI NÓNG ra disk LOCAL (/content) lúc train -> tránh race Drive FUSE (folder mới chưa settle +
# best.pt ghi lại nhiều lần). Cuối mỗi run SYNC sang Drive 1 LẦN (copytree). Leaderboard (runs.csv)
# vẫn là artifact SUY RA từ mọi best.pt trên Drive (rebuild_leaderboard) -> bền với run ngắt + nhiều account.
import json, time, dataclasses, shutil

LOCAL_RUNS = Path('/content/runs_local')               # disk ephemeral của Colab (mất khi hết session, không sao)

def _cfg_from_ckpt(saved_cfg):
    """cfg trong ckpt mang path/device của session train CŨ + có thể THIẾU field mới
    (ckpt v1-v4 pickle theo class cũ). Fill default cho field thiếu, override path/device
    về môi trường hiện tại -> build/eval lại không lỗi."""
    base = config.TwoTowerConfig()
    kw = {f.name: getattr(saved_cfg, f.name, getattr(base, f.name))
          for f in dataclasses.fields(config.TwoTowerConfig)}
    kw['device'] = config.auto_device()
    kw['train_data'] = config.TRAIN_DATA
    return config.TwoTowerConfig(**kw)

def load_run_dir(run_dir):
    """Load best.pt từ 1 thư mục bất kỳ (local hoặc Drive) -> (cfg, ckpt, model, users, logq).
    Dựng model theo cfg LƯU TRONG checkpoint (gồm mọi override d/tau/...) -> khớp shape state_dict;
    chỉ thay path/device về môi trường hiện tại (xem _cfg_from_ckpt)."""
    ckpt = torch.load(Path(run_dir) / 'best.pt', weights_only=False)  # file kèm cfg -> cần False
    cfg = _cfg_from_ckpt(ckpt['cfg'])
    _, logq, _, users, model = train.build(cfg)
    model.load_state_dict(ckpt['model'])
    model.refresh_item_cache()
    return cfg, ckpt, model, users, logq

def load_run(run_name):
    """Load best.pt của 1 run từ Drive (RUNS_DIR/<run_name>) -> sẵn sàng eval/inference."""
    return load_run_dir(RUNS_DIR / run_name)

def eval_run(cfg, model, users, logq):
    """Eval WARM val + test (protocol v2: mask seen−query) -> dict phẳng cho leaderboard.
    Cold slice KHÔNG vào leaderboard (final-exam discipline) — dùng cell 10."""
    out = {}
    seen = data.load_eval_seen(cfg.train_data)
    for split in ['val', 'test']:
        ds = data.ExamplesDataset(cfg.train_data, split)
        q = metrics.group_examples(ds.user_idx, ds.anime_idx)
        masks = metrics.build_masks(seen, q)
        m = metrics.evaluate(model, users, q, logq, cfg.eval_ks, masks,
                             eval_history_cap=cfg.eval_history_cap)
        for k in cfg.eval_ks:
            out[f'{split}_recall@{k}'] = round(float(m[f'recall@{k}']), 4)
            out[f'{split}_ndcg@{k}']   = round(float(m[f'ndcg@{k}']), 4)
    return out

def _make_row(run_name, version, cfg, ckpt, model, users, logq, secs=None, ts=None):
    """1 dòng leaderboard từ (cfg, ckpt, model đã load) — eval lại val+test. Dùng chung run mới + khôi phục."""
    return {
        'run_name': run_name, 'version': version, 'time': ts,
        'd': cfg.d, 'tau': cfg.tau, 'beta': cfg.beta, 'lr': cfg.lr,
        'cosine_lr': cfg.cosine_lr, 'weight_decay': cfg.weight_decay, 'optimizer': cfg.optimizer,
        'batch_size': cfg.batch_size, 'epochs': cfg.epochs,
        'hist_dropout': cfg.hist_dropout, 'm_hardneg': cfg.m_hardneg,
        'use_item_id': cfg.use_item_id, 'id_dim': cfg.id_dim, 'id_dropout': cfg.id_dropout,
        'score_pool': cfg.score_pool,
        'history_pool': cfg.history_pool,
        'history_source': cfg.history_source, 'train_hist_len': cfg.train_hist_len,
        'logq_alpha': cfg.logq_alpha, 'max_examples_per_user': cfg.max_examples_per_user,
        'eval_history_cap': cfg.eval_history_cap,
        'use_synopsis': cfg.use_synopsis, 'synopsis_dim': cfg.synopsis_dim,
        'synopsis_normalize': cfg.synopsis_normalize,
        'train_user_frac': cfg.train_user_frac, 'subset_seed': cfg.subset_seed,
        'best_step': ckpt.get('step'), 'secs': secs,
        **eval_run(cfg, model, users, logq),
    }

def rebuild_leaderboard():
    """runs.csv = gom MỌI run có best.pt (mọi version). Nguồn 1 row, ưu tiên:
    row.json (cache durable) > dòng cũ trong runs.csv (migrate sang row.json) >
    eval lại từ best.pt (run bị ngắt giữa chừng -> dựng row, cache lại). KHÔNG train lại.
    Dedupe theo (version, run_name). Lưu ý: row v1-v4 đo theo protocol CŨ — đừng so
    thẳng với v5 (cột test_recall@200 chỉ có từ v5)."""
    old = {}
    if RESULTS_CSV.exists():
        for r in pd.read_csv(RESULTS_CSV).to_dict('records'):
            old[(str(r.get('version')), str(r.get('run_name')))] = r
    rows, seen = [], set()
    for ckpt_path in sorted((DRIVE / 'runs').glob('*/*/best.pt')):
        run_dir = ckpt_path.parent
        run_name, version = run_dir.name, run_dir.parent.name
        if (version, run_name) in seen:                                 # folder trùng tên -> bỏ qua
            continue
        seen.add((version, run_name))
        row_file = run_dir / 'row.json'
        if row_file.exists():
            row = json.loads(row_file.read_text())
        elif (version, run_name) in old:
            row = old[(version, run_name)]
            row_file.write_text(json.dumps(row, default=float))         # migrate dòng cũ -> durable
        else:                                                           # run bị ngắt -> eval lại từ best.pt
            cfg, ckpt, model, users, logq = load_run_dir(run_dir)
            row = _make_row(run_name, version, cfg, ckpt, model, users, logq)
            row_file.write_text(json.dumps(row, default=float))
            print(f"  recovered {version}/{run_name} (eval từ best.pt)")
        rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_CSV, index=False)
    col = 'test_recall@200' if 'test_recall@200' in df.columns else 'test_recall@100'
    return df.sort_values(col, ascending=False, na_position='last')

def run_experiment(run_name, **overrides):
    """Train ra DISK LOCAL (/content/runs_local) -> tránh race Drive FUSE; cuối run SYNC nguyên
    thư mục sang Drive 1 LẦN (copytree). overrides là field của TwoTowerConfig
    (vd use_item_id=True, id_dim=128, history_source='embed', use_synopsis=True, train_user_frac=0.15)."""
    cfg = dataclasses.replace(base_cfg, **overrides)
    local_dir = LOCAL_RUNS / VERSION / run_name
    local_dir.mkdir(parents=True, exist_ok=True)
    cfg.ckpt_dir = local_dir                            # fit ghi best.pt vào LOCAL (nhanh, không flaky)
    t0 = time.time()
    model, history = train.fit(cfg)
    secs = time.time() - t0
    (local_dir / 'history.json').write_text(json.dumps(history, default=float))
    (local_dir / 'config.json').write_text(json.dumps(dataclasses.asdict(cfg), default=str))  # full cfg = provenance đồ án
    # eval lại từ best.pt LOCAL (model tốt nhất, không phải step cuối) -> row.json
    cfg, ckpt, model_e, users_e, logq_e = load_run_dir(local_dir)
    row = _make_row(run_name, VERSION, cfg, ckpt, model_e, users_e, logq_e,
                    secs=round(secs), ts=pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'))
    (local_dir / 'row.json').write_text(json.dumps(row, default=float))
    # SYNC sang Drive 1 lần: parent RUNS_DIR đã settle từ cell 2 -> tạo 1 folder con (không race)
    RUNS_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_dir, RUNS_DIR / run_name, dirs_exist_ok=True)
    print(f"\n✓ {run_name}: test recall@200={row['test_recall@200']}  (synced -> {RUNS_DIR / run_name})")
    rebuild_leaderboard()                              # runs.csv = gom mọi run (durable, tự lành)
    return cfg, history, row

In [ ]:
# 6) Chạy 1 thử nghiệm. Đổi tên + kwargs cho mỗi lần (checkpoint KHÔNG ghi đè nhau).
# v5 = data v2 + protocol v2. Bar baselines (TEST, warm): MF r@200=.699 ndcg@10=.677,
# ItemKNN .572, Popular .452 — xem PROGRESS.md. Chạy control trước, rồi bật TỪNG đòn bẩy:
cfg, history, row = run_experiment('v5_itemid128_ep2', use_item_id=True, id_dim=128, id_dropout=0.1, epochs=2)

# Đòn bẩy theo thứ tự ưu tiên (mỗi lần đổi 1 thứ so với control):
# cfg, history, row = run_experiment('v5_embed_ep2',  use_item_id=True, id_dim=128, id_dropout=0.1, epochs=2, history_source='embed')
# cfg, history, row = run_experiment('v5_cap64_ep8',  use_item_id=True, id_dim=128, id_dropout=0.1, epochs=8, max_examples_per_user=64)
# cfg, history, row = run_experiment('v5_alpha05_ep2', use_item_id=True, id_dim=128, id_dropout=0.1, epochs=2, logq_alpha=0.5)
# cfg, history, row = run_experiment('v5_hist64_ep2', use_item_id=True, id_dim=128, id_dropout=0.1, epochs=2, train_hist_len=64)

In [ ]:
# 6b) SEARCH hyperparameter — random trên SUBSET (coarse) -> CONFIRM top-K trên FULL.
# run_name TẤT ĐỊNH (encode knob khác default) -> tự dedupe + RESUME khi Colab rớt session
# (skip run đã có trên Drive). Mỗi run vẫn log best.pt/history/row/config + rebuild runs.csv.
# ⚠ space có use_synopsis=True -> CẦN synopsis_emb.npy + synopsis_low_info.npy trong train-data
#   (sinh local: python retriever/data_prep/07_synopsis_emb.py, rồi đẩy 2 .npy lên Drive train-data/).
import search, importlib; importlib.reload(search)

space = {
    'train_hist_len':       [32, 64, 96, 128, 256],
    'history_source':       ['cache', 'embed'],
    'id_dropout':           [0.1, 0.15, 0.2],
    'logq_alpha':           [1.0, 0.75],
    'weight_decay':         [0.0, 1e-5],
    'optimizer':            ['adam', 'adamw'],
    'use_synopsis':         [False, True],
    'synopsis_dim':         [48, 64],
    'synopsis_proj_hidden': [[], [128]],
}
exists = lambda name: (RUNS_DIR / name).exists()       # skip run đã có trên Drive -> resume

# COARSE: subset 15% user + 2 epoch -> xếp hạng RẺ (full catalog+logQ+eval không đổi).
# dry_run=True: chỉ in tên config để xem trước (KHÔNG train). Bỏ dry_run để train thật.
coarse = search.iter_configs(space, method='random', n=12, seed=0,
    fixed={'use_item_id': True, 'id_dim': 128, 'epochs': 2, 'train_user_frac': 0.15})
search.run_search(run_experiment, coarse, exists_fn=exists, dry_run=True)

# CONFIRM (sau khi xem runs.csv lọc theo train_user_frac, chọn top-K): chạy lại FULL data (BỎ train_user_frac).
# cfg, history, row = run_experiment('v5_confirm_hl64_syn48', use_item_id=True, id_dim=128,
#     epochs=2, train_hist_len=64, use_synopsis=True, synopsis_dim=48)

In [ ]:
# 7) Đường cong training loss (raw + moving-average) — dùng `history` trả từ run_experiment
ls = np.array(history['loss_steps']); lv = np.array(history['loss_vals'])
plt.figure(figsize=(8, 4))
plt.plot(ls, lv, alpha=0.3, label='loss')
w = max(1, len(lv) // 50)
if len(lv) >= w and w > 1:
    ma = np.convolve(lv, np.ones(w) / w, mode='valid')
    plt.plot(ls[w - 1:], ma, label=f'MA({w})')
plt.xlabel('step'); plt.ylabel('InfoNCE loss'); plt.title('Training loss')
plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
# 8) Đường cong val metric theo step (recall@K, ndcg@K). Retriever v5 -> để ý recall@200.
es = history['eval_steps']; em = history['eval_metrics']
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for k in cfg.eval_ks:
    ax[0].plot(es, [m[f'recall@{k}'] for m in em], marker='o', label=f'recall@{k}')
    ax[1].plot(es, [m[f'ndcg@{k}'] for m in em], marker='o', label=f'ndcg@{k}')
ax[0].set_title('Val recall@K'); ax[0].set_xlabel('step')
ax[1].set_title('Val ndcg@K'); ax[1].set_xlabel('step')
for a in ax: a.legend(); a.grid(alpha=0.3)
plt.show()

In [ ]:
# 9) Leaderboard: gom MỌI run có best.pt (kể cả run bị ngắt giữa chừng -> eval lại từ best.pt).
# Headline retriever v5 = test_recall@200 (warm). Row v1-v4 đo protocol cũ — đừng so thẳng.
rebuild_leaderboard()

In [ ]:
# 10) COLD SLICE — đo khả năng gợi ý anime mới (H encode id->OOV, content thật).
# KHÔNG nằm trong leaderboard. Discipline: tune bằng split='val'; split='test' là
# FINAL EXAM — chỉ chấm 1 LẦN khi đã chốt model. Bar (test_cold, full-catalog):
# content r@200=.218 hit@500=.378 | meta_popular r@200=.100 | MF/KNN = 0 (PROGRESS.md).
run_name = 'v5_itemid128_ep2'                      # ← đổi theo run cần đo
cfg, ckpt, model, users, logq = load_run(run_name)
for h_only in [False, True]:
    out = metrics.run_cold_eval(model, users, cfg.train_data, logq, cfg.eval_ks,
                                split='val', h_only=h_only)   # 'test' CHỈ khi chấm final
    name = 'H-only (diagnostic content)' if h_only else 'full-catalog'
    print(f"cold {name}: users={out['n_users']:,} pairs={out['n_pairs']:,}")
    print('   ' + '  '.join(f"r@{k}={out[f'recall@{k}']:.4f}" for k in cfg.eval_ks))
    print('   ' + '  '.join(f"hit@{k}={out[f'hitrate@{k}']:.4f}" for k in cfg.eval_ks))
model.refresh_item_cache()   # trả cache về chế độ warm sau khi đo cold